In [28]:
# --- Cell 1: Build focused election dataset file ---
from pathlib import Path
import sys

project_root = Path("/data/disk4/workspace/projects/union")
script_path = project_root / "src" / "build_election_focus_dataset.py"

if not script_path.exists():
    raise FileNotFoundError(f"Script not found: {script_path}")

# Run the focused build script from notebook
exec(script_path.read_text(), {"__name__": "__main__"})
print("\nFocused dataset build completed.")

Reading parquet: /data/disk4/workspace/projects/union/outputs/preliminary_election_level.parquet
Wide shape: (33477, 14107)
Duplicate rows in focus dataset before drop: 0
Focus shape: (33477, 23)
Focus columns: ['election_id', 'case_number', 'election_date', 'tally_type', 'ballot_type', 'runoff_required', 'union_to_certify', 'total_ballots_counted', 'void_ballots', 'challenged_ballots', 'challenges_are_determinative', 'employer_name', 'participant_union_names', 'participant_types_observed', 'unions_involved', 'filing_status', 'filing_date_filed', 'filing_date_closed', 'filing_city', 'filing_state', 'participant_row_count', 'result_row_count', 'tally_row_count']
 election_id  case_number election_date tally_type               ballot_type runoff_required                                  union_to_certify  total_ballots_counted  void_ballots  challenged_ballots challenges_are_determinative                                                                                     employer_name    

In [1]:
# --- Cell 2: Load focused dataset for inspection ---
import pandas as pd
from pathlib import Path

focus_csv = Path("/data/disk4/workspace/projects/union/outputs/preliminary_election_focus.csv")
focus_parquet = Path("/data/disk4/workspace/projects/union/outputs/preliminary_election_focus.parquet")

if focus_parquet.exists():
    df_focus = pd.read_parquet(focus_parquet)
    source_used = str(focus_parquet)
elif focus_csv.exists():
    df_focus = pd.read_csv(focus_csv)
    source_used = str(focus_csv)
else:
    raise FileNotFoundError("Focused output not found. Run Cell 1 first.")

print(f"Loaded from: {source_used}")
print(f"Shape: {df_focus.shape}")
print(f"Columns ({len(df_focus.columns)}):")
print(df_focus.columns.tolist())

Loaded from: /data/disk4/workspace/projects/union/outputs/preliminary_election_focus.parquet
Shape: (33477, 39)
Columns (39):
['election_id', 'case_number', 'election_date', 'tally_type', 'ballot_type', 'runoff_required', 'union_to_certify', 'total_ballots_counted', 'void_ballots', 'challenged_ballots', 'challenges_are_determinative', 'employer_name', 'participant_union_names', 'participant_types_observed', 'unions_involved', 'filing_status', 'filing_date_filed', 'filing_date_closed', 'filing_city', 'filing_state', 'participant_row_count', 'result_row_count', 'tally_row_count', 'votes_for_union', 'votes_against_union', 'total_valid_votes', 'union_support_rate', 'lead_union_name', 'lead_union_votes', 'employer_names_raw', 'employer_types_raw', 'employer_subtypes_raw', 'employer_address_raw', 'employer_address_1_raw', 'employer_address_2_raw', 'employer_city_raw', 'employer_state_raw', 'employer_zip_raw', 'employer_phone_raw']


In [37]:
# --- Election date range + remove elections without vote records ---
from pathlib import Path
import pandas as pd

# 1) Election date range
_date = pd.to_datetime(df_focus['election_date'], errors='coerce')
print(f"Election date range: {_date.min().date()} -> {_date.max().date()}")
print(f"Rows with valid election_date: {_date.notna().sum():,} / {len(df_focus):,}")

# 2) Keep only elections with vote records
# Prefer total_valid_votes if present; fallback to votes_for + votes_against
if 'total_valid_votes' in df_focus.columns:
    valid_vote_mask = pd.to_numeric(df_focus['total_valid_votes'], errors='coerce').fillna(0) > 0
else:
    vf = pd.to_numeric(df_focus.get('votes_for_union', 0), errors='coerce').fillna(0)
    va = pd.to_numeric(df_focus.get('votes_against_union', 0), errors='coerce').fillna(0)
    valid_vote_mask = (vf + va) > 0

df_focus_with_votes = df_focus.loc[valid_vote_mask].copy()

print(f"\nBefore filtering: {len(df_focus):,}")
print(f"After filtering : {len(df_focus_with_votes):,}")
print(f"Removed rows    : {len(df_focus) - len(df_focus_with_votes):,}")

# 3) Export filtered dataset
out_dir = Path('/data/disk4/workspace/projects/union/outputs')
out_dir.mkdir(parents=True, exist_ok=True)

out_csv = out_dir / 'preliminary_election_focus_with_votes.csv'
out_parquet = out_dir / 'preliminary_election_focus_with_votes.parquet'

df_focus_with_votes.to_csv(out_csv, index=False)
try:
    df_focus_with_votes.to_parquet(out_parquet, index=False)
    print(f"Exported: {out_parquet}")
except Exception as e:
    print(f"Parquet export skipped: {e}")

print(f"Exported: {out_csv}")

Election date range: 1994-03-04 -> 2026-04-09
Rows with valid election_date: 33,477 / 33,477

Before filtering: 33,477
After filtering : 31,416
Removed rows    : 2,061
Exported: /data/disk4/workspace/projects/union/outputs/preliminary_election_focus_with_votes.parquet
Exported: /data/disk4/workspace/projects/union/outputs/preliminary_election_focus_with_votes.csv


In [2]:
# --- Next Step 1: keep only RC with vote records ---
import pandas as pd

if 'df_focus_with_votes' not in globals():
    if 'total_valid_votes' in df_focus.columns:
        mask_votes = pd.to_numeric(df_focus['total_valid_votes'], errors='coerce').fillna(0) > 0
    else:
        vf = pd.to_numeric(df_focus.get('votes_for_union', 0), errors='coerce').fillna(0)
        va = pd.to_numeric(df_focus.get('votes_against_union', 0), errors='coerce').fillna(0)
        mask_votes = (vf + va) > 0
    df_focus_with_votes = df_focus.loc[mask_votes].copy()

rc_mask = df_focus_with_votes['case_number'].astype('string').str.contains('-RC-', na=False)
df_rc_votes = df_focus_with_votes.loc[rc_mask].copy()

print(f"RC with votes shape: {df_rc_votes.shape}")
print(df_rc_votes[['election_id', 'case_number', 'election_date']].head(5).to_string(index=False))

RC with votes shape: (26523, 39)
 election_id  case_number election_date
           2 01-RC-020986    1999-04-22
           3 01-RC-021152    2000-04-27
           4 01-RC-021561    2002-11-07
           5 01-RC-022002    2007-04-25
           6 01-RC-022034    2006-11-17


In [3]:
# --- Next Step 2: build employer name candidates (primary + auxiliary) ---
import pandas as pd

name_cols = [c for c in ['employer_name', 'employer_names_raw'] if c in df_rc_votes.columns]
if not name_cols:
    raise RuntimeError('No employer name columns found in df_rc_votes')

emp_long = df_rc_votes[['election_id', 'case_number'] + name_cols].copy()
for c in name_cols:
    emp_long[c] = emp_long[c].astype('string')

# Split pipe-joined names into auxiliary candidates
records = []
for _, r in emp_long.iterrows():
    raw_values = []
    for c in name_cols:
        v = r[c]
        if pd.notna(v):
            raw_values.extend([x.strip() for x in str(v).split(' | ') if x and x.strip()])
    seen = set()
    dedup = []
    for x in raw_values:
        if x not in seen:
            seen.add(x)
            dedup.append(x)
    for i, nm in enumerate(dedup):
        records.append({
            'election_id': r['election_id'],
            'case_number': r['case_number'],
            'employer_candidate_name': nm,
            'candidate_rank': i + 1,
            'is_primary_candidate': i == 0,
        })

df_emp_candidates = pd.DataFrame(records)

print(f"Employer candidate rows: {len(df_emp_candidates):,}")
print(f"Unique candidate names: {df_emp_candidates['employer_candidate_name'].nunique():,}")
print(df_emp_candidates.head(10).to_string(index=False))

Employer candidate rows: 58,028
Unique candidate names: 31,366
 election_id  case_number                                            employer_candidate_name  candidate_rank  is_primary_candidate
           2 01-RC-020986                                      COURTYARD NURSING CARE CENTER               1                  True
           3 01-RC-021152                                    STEWARD CARITAS CARNEY HOSPITAL               1                  True
           4 01-RC-021561                                                    TELCOM USA INC.               1                  True
           5 01-RC-022002          C. BROOKS & DOUGLAS P. CURRIER, ROBERT\nVerrill Dana, LLP               1                  True
           5 01-RC-022002                              REGIONAL TRANSPORTATION PROGRAM, INC.               2                 False
           6 01-RC-022034 P JEWELER, BERNARD\nOgletree, Deakins, Nash, Smoak & Stewart, P.C.               1                  True
           6 01-RC-0

In [4]:
# --- Next Step 3: load Compustat company names from WRDS (schema-robust) ---
import pandas as pd
import wrds

WRDS_USERNAME = "wangyouan"
db = wrds.Connection(wrds_username=WRDS_USERNAME)

# 1) Inspect available columns in comp.names
cols_sql = """
SELECT column_name
FROM information_schema.columns
WHERE table_schema = 'comp' AND table_name = 'names'
ORDER BY ordinal_position
"""
cols_df = db.raw_sql(cols_sql)
avail_cols = cols_df['column_name'].tolist()
print(f"comp.names available columns: {len(avail_cols)}")

# 2) Build dynamic select list
preferred = ['gvkey', 'conm', 'tic', 'cik', 'cusip', 'fic', 'loc', 'naics', 'sic']
select_cols = [c for c in preferred if c in avail_cols]
if 'gvkey' not in select_cols or 'conm' not in select_cols:
    raise RuntimeError(f"Required columns missing in comp.names. available={avail_cols}")

query = f"SELECT {', '.join(select_cols)} FROM comp.names WHERE conm IS NOT NULL"
df_comp = db.raw_sql(query)

# 3) Matching name field
df_comp['comp_name_for_match'] = df_comp['conm'].astype('string')
df_comp = df_comp.dropna(subset=['comp_name_for_match']).drop_duplicates(subset=['gvkey', 'comp_name_for_match']).copy()

print(f"Compustat rows loaded: {len(df_comp):,}")
show_cols = [c for c in ['gvkey', 'comp_name_for_match', 'tic', 'sic', 'naics'] if c in df_comp.columns]
print(df_comp[show_cols].head(10).to_string(index=False))

Loading library list...
Done
comp.names available columns: 11
Compustat rows loaded: 47,101
 gvkey      comp_name_for_match    tic  sic  naics
001000    A & E PLASTIK PAK INC   AE.2 3089   <NA>
001001  A & M FOOD SERVICES INC  AMFD. 5812    722
001002                 AAI CORP AAIC.1 3825   <NA>
001003    A.A. IMPORTING CO INC   ANTQ 5712 442110
001004                 AAR CORP    AIR 5080 423860
001005    A.B.A. INDUSTRIES INC  ABA.2 3724   <NA>
001006             ABC INDS INC  1145B 2711   <NA>
001007     ABKCO INDUSTRIES INC  4135B 3652 334612
001008 ABM COMPUTER SYSTEMS INC  ABMC. 3577 334119
001009       ABS INDUSTRIES INC ABSI.1 3460   3321


In [7]:
# --- Next Step 4: fuzzy match employer candidates to Compustat names (clean + hybrid) ---
import re
import pandas as pd
from rapidfuzz import fuzz, process
from name_matching.name_matcher import NameMatcher

# 1) Unique query names
q = df_emp_candidates[['employer_candidate_name']].drop_duplicates().copy()
q['employer_candidate_name'] = q['employer_candidate_name'].astype('string').str.strip()
q = q[q['employer_candidate_name'].notna() & (q['employer_candidate_name'] != '')].copy()
q = q.reset_index(drop=True)

# 2) Build reference table with stable row index for mapping
meta_cols_preferred = ['tic', 'cik', 'cusip', 'fic', 'loc', 'naics', 'sic']
meta_cols = [c for c in meta_cols_preferred if c in df_comp.columns]
ref_cols = ['gvkey', 'comp_name_for_match'] + meta_cols

r = df_comp[ref_cols].dropna(subset=['comp_name_for_match']).copy()
r['comp_name_for_match'] = r['comp_name_for_match'].astype('string').str.strip()
r = r[r['comp_name_for_match'] != ''].reset_index(drop=True)
r = r.reset_index().rename(columns={'index': 'master_row_id'})


def normalize_company_name(x: str) -> str:
    if x is None:
        return ''
    s = str(x).upper().replace('\n', ' ').strip()
    s = re.sub(r'[^A-Z0-9 ]+', ' ', s)
    s = re.sub(r'\b(INC|INCORPORATED|CORP|CORPORATION|CO|COMPANY|LLC|LTD|LIMITED|PLC|LP|LLP|PC)\b', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s


q['query_for_match'] = q['employer_candidate_name'].map(normalize_company_name)
q = q[q['query_for_match'].str.len() > 1].copy().reset_index(drop=True)

r['ref_for_match'] = r['comp_name_for_match'].map(normalize_company_name)
r = r[r['ref_for_match'].str.len() > 1].copy().reset_index(drop=True)

print(f"Candidate names after normalization: {len(q):,}")

# 3) Track A: name_matching
matcher = NameMatcher(
    number_of_matches=1,
    legal_suffixes=True,
    common_words=False,
    top_n=50,
    verbose=False,
)
matcher.set_distance_metrics(['bag', 'typo', 'refined_soundex'])
matcher.load_and_process_master_data(
    column='ref_for_match',
    df_matching_data=r[['master_row_id', 'ref_for_match']],
    transform=True,
)

nm_matches = matcher.match_names(
    to_be_matched=q[['query_for_match']],
    column_matching='query_for_match',
)

nm_res = q.reset_index().rename(columns={'index': 'q_row_id'})
nm_res = nm_res.merge(nm_matches.reset_index().rename(columns={'index': 'q_row_id'}), on='q_row_id', how='left')

nm_score_col = next((c for c in ['score', 'match_score', 'distance_score'] if c in nm_res.columns), None)
nm_idx_col = next((c for c in ['match_index', 'index_match', 'master_index', 'index_0'] if c in nm_res.columns), None)
if nm_idx_col is None:
    idx_candidates = [c for c in nm_res.columns if 'index' in c.lower() and c != 'q_row_id']
    nm_idx_col = idx_candidates[0] if idx_candidates else None
if nm_score_col is None:
    score_candidates = [c for c in nm_res.columns if 'score' in c.lower() or 'sim' in c.lower()]
    nm_score_col = score_candidates[0] if score_candidates else None
if nm_idx_col is None:
    raise RuntimeError(f'Cannot find name_matching index column. columns={nm_res.columns.tolist()}')
if nm_score_col is None:
    nm_res['nm_score'] = pd.NA
    nm_score_col = 'nm_score'

nm_res[nm_idx_col] = pd.to_numeric(nm_res[nm_idx_col], errors='coerce')
nm_res[nm_score_col] = pd.to_numeric(nm_res[nm_score_col], errors='coerce')

nm_res = nm_res.merge(
    r[['master_row_id', 'gvkey', 'comp_name_for_match'] + meta_cols],
    left_on=nm_idx_col,
    right_on='master_row_id',
    how='left',
)

nm_out = pd.DataFrame({
    'employer_candidate_name': nm_res['employer_candidate_name'],
    'nm_match_score': nm_res[nm_score_col],
    'nm_matched_gvkey': nm_res['gvkey'],
    'nm_matched_conm': nm_res['comp_name_for_match'],
})
for c in meta_cols:
    nm_out[f'nm_matched_{c}'] = nm_res[c]

# 4) Track B: rapidfuzz (blocked)
ref_by_first = {}
for _, rr in r[['master_row_id', 'ref_for_match']].iterrows():
    key = rr['ref_for_match'][0]
    ref_by_first.setdefault(key, []).append((rr['master_row_id'], rr['ref_for_match']))

all_pool = list(zip(r['master_row_id'].tolist(), r['ref_for_match'].tolist()))

rf_records = []
for _, row in q[['employer_candidate_name', 'query_for_match']].iterrows():
    query_name = row['query_for_match']
    first = query_name[0]
    pool = ref_by_first.get(first) or all_pool

    pool_names = [x[1] for x in pool]
    hit = process.extractOne(query_name, pool_names, scorer=fuzz.token_sort_ratio)
    if hit is None:
        rf_records.append({
            'employer_candidate_name': row['employer_candidate_name'],
            'rf_match_score': pd.NA,
            'rf_master_row_id': pd.NA,
        })
        continue

    _, matched_score, matched_pos = hit
    matched_master_row_id = pool[matched_pos][0]
    rf_records.append({
        'employer_candidate_name': row['employer_candidate_name'],
        'rf_match_score': float(matched_score),
        'rf_master_row_id': int(matched_master_row_id),
    })

rf_out = pd.DataFrame(rf_records)
rf_out = rf_out.merge(
    r[['master_row_id', 'gvkey', 'comp_name_for_match'] + meta_cols],
    left_on='rf_master_row_id',
    right_on='master_row_id',
    how='left',
)
rf_out = rf_out.rename(columns={'gvkey': 'rf_matched_gvkey', 'comp_name_for_match': 'rf_matched_conm'})
for c in meta_cols:
    rf_out = rf_out.rename(columns={c: f'rf_matched_{c}'})

# 5) Hybrid pick: choose higher score between nm and rf
hyb = nm_out.merge(rf_out, on='employer_candidate_name', how='outer')

hyb['nm_match_score'] = pd.to_numeric(hyb['nm_match_score'], errors='coerce')
hyb['rf_match_score'] = pd.to_numeric(hyb['rf_match_score'], errors='coerce')

use_rf = hyb['rf_match_score'].fillna(-1) > hyb['nm_match_score'].fillna(-1)

hyb['match_score'] = hyb['nm_match_score']
hyb.loc[use_rf, 'match_score'] = hyb.loc[use_rf, 'rf_match_score']

hyb['matched_gvkey'] = hyb['nm_matched_gvkey']
hyb['matched_conm'] = hyb['nm_matched_conm']
hyb.loc[use_rf, 'matched_gvkey'] = hyb.loc[use_rf, 'rf_matched_gvkey']
hyb.loc[use_rf, 'matched_conm'] = hyb.loc[use_rf, 'rf_matched_conm']

for c in meta_cols:
    nm_col = f'nm_matched_{c}'
    rf_col = f'rf_matched_{c}'
    out_col = f'matched_{c}'
    hyb[out_col] = hyb[nm_col] if nm_col in hyb.columns else pd.NA
    if rf_col in hyb.columns:
        hyb.loc[use_rf, out_col] = hyb.loc[use_rf, rf_col]

df_name_match = hyb[['employer_candidate_name', 'match_score', 'matched_gvkey', 'matched_conm'] + [f'matched_{c}' for c in meta_cols if f'matched_{c}' in hyb.columns]].copy()
df_name_match = df_name_match.drop_duplicates(subset=['employer_candidate_name'], keep='first')

print(f"Name-level matches built (hybrid): {len(df_name_match):,}")
print(f"Using rapidfuzz winner rows: {int(use_rf.fillna(False).sum()):,}")
print(df_name_match.sort_values('match_score', ascending=False).head(10).to_string(index=False))

Candidate names after normalization: 31,359
Name-level matches built (hybrid): 31,359
Using rapidfuzz winner rows: 28,076
        employer_candidate_name  match_score matched_gvkey               matched_conm matched_tic matched_cik matched_cusip matched_naics matched_sic
   H&E EQUIPMENT SERVICES, INC.        100.0        165856 H&E EQUIPMENT SERVICES INC        HEES  0001339605     404030108        423830        5084
***, ***\nStarbucks Corporation        100.0        025434             STARBUCKS CORP        SBUX  0000829224     855244109        722513        5812
                      Zep, Inc.        100.0        178538                    ZEP INC         ZEP  0001408287     98944B108        325612        2842
               Albertson's LLC.        100.0        001240            ALBERTSON'S INC       ABS.1  0000003333     013104104        445110        5411
              Albertson's, Inc.        100.0        001240            ALBERTSON'S INC       ABS.1  0000003333     013104104     

In [8]:
# --- Next Step 5: collapse candidate matches to election-level best match ---
import pandas as pd
from pathlib import Path

# Join candidate-level matches back
df_emp_match = df_emp_candidates.merge(df_name_match, on='employer_candidate_name', how='left')

# For each election_id, keep best match score across primary + auxiliary employer names
df_best = (
    df_emp_match.sort_values(['election_id', 'match_score'], ascending=[True, False])
    .drop_duplicates(subset=['election_id'], keep='first')
    .copy()
)

keep_cols = [
    'election_id', 'case_number', 'employer_candidate_name', 'candidate_rank', 'is_primary_candidate',
    'match_score', 'matched_gvkey', 'matched_conm', 'matched_tic', 'matched_cik', 'matched_cusip',
    'matched_fic', 'matched_loc', 'matched_naics', 'matched_sic'
]
keep_cols = [c for c in keep_cols if c in df_best.columns]
df_best = df_best[keep_cols]

# Merge into RC vote sample
df_rc_votes_matched = df_rc_votes.merge(df_best, on=['election_id', 'case_number'], how='left')

print(f"Matched RC sample shape: {df_rc_votes_matched.shape}")
print(f"Rows with gvkey match: {df_rc_votes_matched['matched_gvkey'].notna().sum():,}")
print(f"Rows with score>=90: {(df_rc_votes_matched['match_score'] >= 90).fillna(False).sum():,}")
print(df_rc_votes_matched[['election_id', 'case_number', 'employer_name', 'matched_conm', 'matched_gvkey', 'match_score']].head(12).to_string(index=False))

out_dir = Path('/data/disk4/workspace/projects/union/outputs')
out_dir.mkdir(parents=True, exist_ok=True)
out_csv = out_dir / 'preliminary_election_focus_with_votes_rc_compustat_match.csv'
out_parquet = out_dir / 'preliminary_election_focus_with_votes_rc_compustat_match.parquet'

df_rc_votes_matched.to_csv(out_csv, index=False)
try:
    df_rc_votes_matched.to_parquet(out_parquet, index=False)
except Exception as e:
    print(f"Parquet export skipped: {e}")

print(f"Exported: {out_csv}")
print(f"Exported: {out_parquet}")

Matched RC sample shape: (26523, 50)
Rows with gvkey match: 26,422
Rows with score>=90: 2,558
 election_id  case_number                                                                                                                                                                                       employer_name                 matched_conm matched_gvkey  match_score
           2 01-RC-020986                                                                                                                                                                       COURTYARD NURSING CARE CENTER CONTINENTAL CARE CENTERS INC        003457    64.150943
           3 01-RC-021152                                                                                                                                                                     STEWARD CARITAS CARNEY HOSPITAL  SHEPARD NILES CRANE & HOIST        009661    60.714286
           4 01-RC-021561                                               

In [15]:
# --- Match summary (compact) ---
import pandas as pd

if 'df_rc_votes_matched' not in globals():
    raise RuntimeError('df_rc_votes_matched not found. Run matching cells first.')

total = len(df_rc_votes_matched)
matched = int(df_rc_votes_matched['matched_gvkey'].notna().sum())
share = matched / total if total else 0

s = pd.to_numeric(df_rc_votes_matched['match_score'], errors='coerce')
print(f'total RC-with-votes rows: {total:,}')
print(f'matched rows (gvkey not null): {matched:,}')
print(f'match rate: {share:.2%}')
print(f'match score >= 95: {(s >= 95).fillna(False).sum():,}')
print(f'match score >= 90: {(s >= 90).fillna(False).sum():,}')
print(f'match score >= 85: {(s >= 85).fillna(False).sum():,}')
print(f'match score >= 80: {(s >= 80).fillna(False).sum():,}')

# --- Score >= 80: elections vs. company-year breakdown ---
df80 = df_rc_votes_matched[s.fillna(-1) >= 80].copy()
df80['election_year'] = pd.to_datetime(df80['election_date'], errors='coerce').dt.year

n_elections = len(df80)
n_unique_gvkey = df80['matched_gvkey'].nunique()
n_company_year = df80.dropna(subset=['matched_gvkey', 'election_year']).drop_duplicates(subset=['matched_gvkey', 'election_year']).shape[0]

print(f'\n--- Score >= 80 breakdown ---')
print(f'Elections:                   {n_elections:,}')
print(f'Unique companies (gvkey):    {n_unique_gvkey:,}')
print(f'Unique company-year obs:     {n_company_year:,}')
print(f'Avg elections per company:   {n_elections / n_unique_gvkey:.2f}')
print(f'Avg elections per co-year:   {n_elections / n_company_year:.2f}')


# ============================================================
# --- Supplement with old match file (score < 80 fallback) ---
# ============================================================
import pandas as pd
from pathlib import Path

# 1) Load old match file
old_path = '/data/disk4/workspace/datasets_processed/union/20220319_union_election_merge_with_gvkey.pkl'
df_old = pd.read_pickle(old_path)

# Normalize join key and old gvkey
df_old = df_old.copy()
df_old['case_number'] = df_old['casenumber'].astype('string').str.strip()
df_old['gvkey_old'] = df_old['gvkey'].apply(
    lambda x: str(int(x)).zfill(6) if pd.notna(x) and x > 0 else pd.NA
)

old_lookup = df_old[['case_number', 'gvkey_old']].dropna(subset=['gvkey_old']).drop_duplicates(subset=['case_number'], keep='first')
print(f"\nOld file: {len(df_old):,} rows, {old_lookup['case_number'].nunique():,} cases with gvkey")

# 2) Build base: all RC-with-votes rows
df_new = df_rc_votes_matched.copy()
df_new['case_number'] = df_new['case_number'].astype('string').str.strip()
df_new['match_score_num'] = pd.to_numeric(df_new['match_score'], errors='coerce')

# New match: score >= 80 → accepted
new_high_conf = df_new['match_score_num'].fillna(-1) >= 80

# 3) Merge old gvkey for fallback
df_new = df_new.merge(old_lookup, on='case_number', how='left')

# 4) Decide final gvkey
df_new['gvkey_final'] = pd.NA
df_new['gvkey_source'] = pd.NA

# New high-confidence wins
df_new.loc[new_high_conf, 'gvkey_final'] = df_new.loc[new_high_conf, 'matched_gvkey']
df_new.loc[new_high_conf, 'gvkey_source'] = 'new_match'

# Old file supplements where new match is below threshold
old_fill_mask = ~new_high_conf & df_new['gvkey_old'].notna()
df_new.loc[old_fill_mask, 'gvkey_final'] = df_new.loc[old_fill_mask, 'gvkey_old']
df_new.loc[old_fill_mask, 'gvkey_source'] = 'old_match'

# 5) Summary
n_total = len(df_new)
n_new = int((df_new['gvkey_source'] == 'new_match').sum())
n_old = int((df_new['gvkey_source'] == 'old_match').sum())
n_unmatched = int(df_new['gvkey_final'].isna().sum())

print(f"\n--- Combined match summary ---")
print(f"Total elections:             {n_total:,}")
print(f"From new match (score>=80):  {n_new:,}")
print(f"Supplemented from old file:  {n_old:,}")
print(f"Still unmatched:             {n_unmatched:,}")
print(f"Total with gvkey:            {n_total - n_unmatched:,}  ({(n_total - n_unmatched)/n_total:.2%})")

# 6) Company-year breakdown for combined sample
df_combined = df_new[df_new['gvkey_final'].notna()].copy()
df_combined['election_year'] = pd.to_datetime(df_combined['election_date'], errors='coerce').dt.year
cy = df_combined.dropna(subset=['gvkey_final', 'election_year']).drop_duplicates(subset=['gvkey_final', 'election_year'])
print(f"\nCombined matched unique companies:   {df_combined['gvkey_final'].nunique():,}")
print(f"Combined unique company-year obs:    {len(cy):,}")


# 7) Export
out_dir = Path('/data/disk4/workspace/projects/union/outputs')
out_dir.mkdir(parents=True, exist_ok=True)

out_parquet = out_dir / 'union_election_rc_votes_matched_combined.parquet'
out_csv = out_dir / 'union_election_rc_votes_matched_combined.csv'

df_new.to_parquet(out_parquet, index=False)
df_new.to_csv(out_csv, index=False)
print(f"\nExported: {out_parquet}")
print(f"Exported: {out_csv}")


total RC-with-votes rows: 26,523
matched rows (gvkey not null): 26,422
match rate: 99.62%
match score >= 95: 2,180
match score >= 90: 2,558
match score >= 85: 3,320
match score >= 80: 4,660

--- Score >= 80 breakdown ---
Elections:                   4,660
Unique companies (gvkey):    1,566
Unique company-year obs:     2,867
Avg elections per company:   2.98
Avg elections per co-year:   1.63

Old file: 34,462 rows, 3,949 cases with gvkey

--- Combined match summary ---
Total elections:             26,523
From new match (score>=80):  4,660
Supplemented from old file:  246
Still unmatched:             21,617
Total with gvkey:            4,906  (18.50%)

Combined matched unique companies:   1,635
Combined unique company-year obs:    3,023

Exported: /data/disk4/workspace/projects/union/outputs/union_election_rc_votes_matched_combined.parquet
Exported: /data/disk4/workspace/projects/union/outputs/union_election_rc_votes_matched_combined.csv
